# "Có ý nghĩa thống kê" chưa phải là "đáng kể": correlation và t-test trên Bike Sharing

Bạn là DA của hệ thống xe đạp công cộng Capital Bikeshare (Washington DC, bộ Bike Sharing UCI, `day.csv`,
731 ngày 2011-2012). Hai câu hỏi tới cùng lúc:

- **Trưởng bộ phận vận hành**: "Thời tiết ảnh hưởng tới lượt thuê cỡ nào? Nhóm dữ liệu nói độ ẩm *có ý nghĩa
  thống kê*, vậy ngày ẩm có nên giảm xe không?"
- **Trưởng marketing**: "Ngày lễ có làm lượt thuê giảm không? Nếu có thì ngày lễ nên chạy khuyến mãi cho
  khách vãng lai."

Bạn trả lời bằng **độ lớn kèm khoảng tin cậy**, không chỉ p-value, và kiểm xem so sánh có công bằng không (ngày
lễ rơi vào mùa nào?).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/11-stats-experimentation/exercises/basic/03-bike-significance-vs-size -q`

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

## Phần A. Correlation có khoảng tin cậy

`to_business_units(day) -> DataFrame`: bản sao của `day` thêm ba cột theo đơn vị người vận hành hiểu (theo
Readme của UCI: nhiệt độ chuẩn hoá chia cho 41, độ ẩm chia 100, gió chia 67):
`temp_c = temp * 41` (°C), `hum_pct = hum * 100` (%), `wind_kmh = windspeed * 67` (km/h).

`corr_with_ci(x, y, alpha=0.05) -> dict` với khoá `r`, `ci_low`, `ci_high`, `p`, `n`:

- `r`: hệ số Pearson.
- Khoảng tin cậy bằng **Fisher z**: $$z = \operatorname{artanh}(r)$$, sai số chuẩn $$1/\sqrt{n-3}$$,
  khoảng $$\tanh\left(z \pm z_{1-\alpha/2} / \sqrt{n-3}\right)$$.
- `p`: hai phía, từ $$t = r\sqrt{(n-2)/(1-r^2)}$$ với phân phối t bậc tự do $$n-2$$.
- Bỏ các cặp có giá trị thiếu trước khi tính; `n` là số cặp còn lại.

In [ ]:
def to_business_units(day: pd.DataFrame) -> pd.DataFrame:
    """Bản sao thêm temp_c (°C), hum_pct (%), wind_kmh (km/h)."""
    # TODO
    raise NotImplementedError


def corr_with_ci(x: pd.Series, y: pd.Series, alpha: float = 0.05) -> dict:
    """Pearson r, khoảng tin cậy Fisher z, p hai phía, n."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: đọc day.csv, correlation của ba biến thời tiết với lượt thuê
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure  # noqa: E402

day = to_business_units(pd.read_csv(ensure("bike-sharing-uci") / "day.csv", parse_dates=["dteday"]))
print(len(day), "ngày,", f"{day.cnt.sum():,}", "lượt")
rows = {col: corr_with_ci(day[col], day["cnt"]) for col in ["temp_c", "hum_pct", "wind_kmh"]}
table = pd.DataFrame(rows).T
table["r2"] = table["r"] ** 2
print(table[["r", "ci_low", "ci_high", "p", "r2"]].round(4).to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, col, unit in zip(axes, ["temp_c", "hum_pct", "wind_kmh"], ["°C", "%", "km/h"]):
    ax.scatter(day[col], day["cnt"], s=6, alpha=0.4, color="#4c78a8")
    res = rows[col]
    ax.set_title(f"r = {res['r']:.2f} [{res['ci_low']:.2f}; {res['ci_high']:.2f}]", loc="left", fontsize=10)
    ax.set_xlabel(f"{col} ({unit})")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("lượt thuê / ngày")
fig.suptitle("Độ ẩm có p < 0.01 nhưng gần như không giải thích được gì", x=0.01, ha="left")
fig.tight_layout()
plt.show()

## Phần B. Ngày lễ: so sánh hai nhóm và so sánh công bằng

`welch_compare(a, b, alpha=0.05) -> dict` với khoá `mean_a`, `mean_b`, `diff` (`mean_a - mean_b`), `ci_low`,
`ci_high`, `p`, `n_a`, `n_b`: Welch t-test (không giả định hai nhóm cùng phương sai), hai phía. Sai số chuẩn
$$\sqrt{s_a^2/n_a + s_b^2/n_b}$$ (phương sai mẫu, `ddof=1`), bậc tự do Welch-Satterthwaite:

$$\nu = \frac{\left(s_a^2/n_a + s_b^2/n_b\right)^2}{\frac{(s_a^2/n_a)^2}{n_a-1} + \frac{(s_b^2/n_b)^2}{n_b-1}}$$

khoảng tin cậy `diff ± t(1 - alpha/2, ν) * se`, `p` từ phân phối t bậc tự do `ν`.

`matched_holiday_gap(day, col) -> Series`: với **mỗi ngày lễ** (`holiday == 1`), lấy giá trị `col` của ngày đó
trừ trung bình `col` của các **ngày làm việc** (`workingday == 1`) cùng năm và cùng tháng (`yr`, `mnth`).
Index là `dteday` của ngày lễ (theo thứ tự trong `day`), tên `gap`. Đây là cách so "cùng mùa với cùng mùa".

In [ ]:
def welch_compare(a, b, alpha: float = 0.05) -> dict:
    """Welch t-test hai phía: mean_a, mean_b, diff, ci_low, ci_high, p, n_a, n_b."""
    # TODO
    raise NotImplementedError


def matched_holiday_gap(day: pd.DataFrame, col: str) -> pd.Series:
    """Giá trị ngày lễ trừ trung bình ngày làm việc cùng yr, mnth. Index dteday, tên "gap"."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: ngày lễ so với ngày làm việc: so thẳng, rồi so cùng tháng
holiday, work = day[day.holiday == 1], day[day.workingday == 1]
print("ngày lễ rơi vào tháng:", holiday.mnth.value_counts().sort_index().to_dict())
raw = pd.DataFrame({col: welch_compare(holiday[col], work[col]) for col in ["cnt", "casual", "registered"]}).T
print("So thẳng (lễ - ngày làm việc):")
print(raw[["mean_a", "mean_b", "diff", "ci_low", "ci_high", "p"]].round(3).to_string())

matched = {}
for col in ["cnt", "casual", "registered"]:
    gap = matched_holiday_gap(day, col)
    res = stats.ttest_1samp(gap, 0)
    ci = res.confidence_interval()
    matched[col] = {"mean_gap": gap.mean(), "ci_low": ci.low, "ci_high": ci.high, "p": res.pvalue,
                    "holidays_below": int((gap < 0).sum())}
print("\nSo cùng năm-tháng (21 ngày lễ):")
print(pd.DataFrame(matched).T.round(3).to_string())

gap = matched_holiday_gap(day, "cnt")
fig, ax = plt.subplots(figsize=(10, 3.5))
colors = np.where(gap.values < 0, "#e45756", "#4c78a8")
ax.bar([d.strftime("%Y-%m-%d") for d in gap.index], gap.values, color=colors)
ax.axhline(0, color="#333", lw=0.8)
ax.tick_params(axis="x", rotation=90, labelsize=7)
ax.set_ylabel("lượt so với ngày làm việc cùng tháng")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"Ngày lễ trung bình {gap.mean():+.0f} lượt so với ngày làm việc cùng tháng", loc="left")
plt.show()

## Đọc kết quả, rồi trả lời hai trưởng bộ phận

1. Độ ẩm: p-value bao nhiêu, r² bao nhiêu? "Có ý nghĩa thống kê" nói gì và không nói gì về việc điều xe?
2. Nhiệt độ: khoảng tin cậy của r hẹp hay rộng? Vì sao correlation này **chưa** đủ để nói "trời ấm làm tăng
   lượt thuê X%"? (gợi ý: mùa, năm, xem phần regression ở bài Intermediate 02)
3. Ngày lễ: so thẳng cho kết luận gì với `cnt` (p bao nhiêu)? So cùng tháng thì sao? Vì sao hai cách khác
   nhau? Ngày lễ rơi vào những tháng nào?
4. `casual` và `registered` đi ngược chiều nhau trong ngày lễ. Câu trả lời cho trưởng marketing là gì?
5. Bạn đã kiểm 3 biến thời tiết và 3 chỉ số ngày lễ: 6 kiểm định. Điều đó ảnh hưởng thế nào tới cách đọc
   một p-value quanh 0.04?

Xong thì mở app, làm phần **trắc nghiệm** của bài.